# **Milestone 2**

## **Introduction**

This milestone has been created to transition you from classical NLP techniques to modern, state-of-the-art deep learning architectures. It focuses on familiarizing you with the Hugging Face ecosystem, understanding how attention mechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to drastically improve upon your baseline ranking metrics.  

Suggested Readings:
* Hugging Face transformers & datasets Library Basics  

* The Attention Mechanism and BERT/RoBERTa Architectures

* Dense Context-Aware Embeddings (e.g., Sentence-Transformers)

* Zero-Shot Classification

* Softmax vs. Independent Sigmoid (Multi-label) Probabilities  

* Prompting Small Language Models (SLMs) for Generative QA

Perform the following tasks on the dataset provided as a part of the Kaggle competition.
Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

## **Initialization**

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
sample_df = pd.read_csv("/content/sample_submission.csv")

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## **Question 01**

Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [5]:
dataset = load_dataset("csv", data_files={"train": "/content/train.csv"})["train"]

def combine_text(example):
    example["combined_text"] = str(example["prompt"]) + " " + str(example["A"])
    return example

dataset = dataset.map(combine_text)
print(len(dataset[51]["combined_text"]))

614


## **Question 02**

Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

30522


## **Question 03**

Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [7]:
print(tokenizer.sep_token_id)

102


## **Question 04**

Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [9]:
prompts = [str(x) for x in dataset["prompt"]]
tokens = tokenizer(prompts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
print(tokens["input_ids"].shape)

torch.Size([2000, 128])


## **Question 05**

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?

In [10]:
hidden_size = 768
num_heads = 12
print(hidden_size // num_heads)

64


## **Question 06**

Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

Note: We follow zero-indexing here.

In [12]:
model = AutoModel.from_pretrained("bert-base-uncased")
prompt_0 = str(dataset[0]["prompt"])
tokens_0 = tokenizer(prompt_0, return_tensors="pt")

with torch.no_grad():
    outputs = model(**tokens_0)

last_hidden_state = outputs.last_hidden_state
print(last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


## **Question 07**

Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).

In [15]:
cls_embedding = last_hidden_state[0, 0, :]
sum_first_5 = torch.sum(cls_embedding[:5]).item()
print(round(sum_first_5, 4))

-1.2001


## **Question 08**

Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [17]:
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
text = "Light-ion fusion is a technique."
tokens = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs_attn = model_attn(**tokens)

attentions = outputs_attn.attentions
last_layer_attn = attentions[-1]
first_head_attn = last_layer_attn[0, 0, :, :]

tokens_list = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])
fusion_index = tokens_list.index("fusion")

attention_weight = first_head_attn[0, fusion_index].item()
print(round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025


## **Question 09**

Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [19]:
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
prompt_0 = str(dataset[0]["prompt"])
option_b_0 = str(dataset[0]["B"])

emb_prompt = st_model.encode(prompt_0, convert_to_tensor=True)
emb_option_b = st_model.encode(option_b_0, convert_to_tensor=True)

sim = util.cos_sim(emb_prompt, emb_option_b).item()
print(round(sim, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.7658


## **Question 10**

Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?

In [21]:
train_df = pd.read_csv("/content/train.csv")
all_text = []
for col in ["prompt", "A", "B", "C", "D", "E"]:
    all_text.extend(train_df[col].dropna().astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_text)

def map_at_3(truth, preds):
    if truth == preds[0]:
        return 1.0
    elif truth == preds[1]:
        return 1/2
    elif truth == preds[2]:
        return 1/3
    return 0.0

option_letters = ["A", "B", "C", "D", "E"]
tfidf_top3_list = []
minilm_top3_list = []
map3_minilm_scores = []

prompts_text = train_df["prompt"].astype(str).tolist()
options_text = {
    "A": train_df["A"].astype(str).tolist(),
    "B": train_df["B"].astype(str).tolist(),
    "C": train_df["C"].astype(str).tolist(),
    "D": train_df["D"].astype(str).tolist(),
    "E": train_df["E"].astype(str).tolist()
}

emb_prompts = st_model.encode(prompts_text, convert_to_tensor=True)
emb_options = {
    opt: st_model.encode(options_text[opt], convert_to_tensor=True) for opt in option_letters
}

for i in range(len(train_df)):
    truth = train_df.iloc[i]["answer"]

    prompt_str = prompts_text[i]
    opts_str = [options_text[opt][i] for opt in option_letters]
    tfidf_matrix = vectorizer.transform([prompt_str] + opts_str)
    tfidf_sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:6])[0]
    sorted_tfidf = [opt for _, opt in sorted(zip(tfidf_sims, option_letters), reverse=True)]
    tfidf_top3_list.append(sorted_tfidf[:3])

    minilm_sims = [util.cos_sim(emb_prompts[i], emb_options[opt][i]).item() for opt in option_letters]
    sorted_minilm = [opt for _, opt in sorted(zip(minilm_sims, option_letters), reverse=True)]
    minilm_top3 = sorted_minilm[:3]
    minilm_top3_list.append(minilm_top3)

    map3_minilm_scores.append(map_at_3(truth, minilm_top3))

print("Final MAP@3 score of all-MiniLM-L6-v2 pipeline:", np.mean(map3_minilm_scores))

count_diff = 0
for i in range(len(train_df)):
    truth = train_df.iloc[i]["answer"]
    if truth not in tfidf_top3_list[i] and truth in minilm_top3_list[i]:
        count_diff += 1

print("Count where correct answer is NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3:", count_diff)

Final MAP@3 score of all-MiniLM-L6-v2 pipeline: 0.4230833333333333
Count where correct answer is NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3: 575


## **Question 11**

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [23]:
zero_shot_clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
prompt_1 = str(dataset[1]["prompt"])
options_1 = [str(dataset[1]["A"]), str(dataset[1]["B"]), str(dataset[1]["C"])]

result_q11 = zero_shot_clf(prompt_1, candidate_labels=options_1)
top_score_q11 = result_q11["scores"][0]
print(round(top_score_q11, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

0.4575


## **Question 12**

Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True.

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [24]:
result_q12 = zero_shot_clf(prompt_1, candidate_labels=options_1, multi_label=True)
sum_q11 = sum(result_q11["scores"])
sum_q12 = sum(result_q12["scores"])

abs_diff = abs(sum_q11 - sum_q12)
print(abs_diff)

0.9994903945043916


## **Question 13**

Let's try Generative AI instead of Classification.

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [26]:
tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt_0_str = str(dataset[0]["prompt"])
opt_A_0 = str(dataset[0]["A"])
opt_B_0 = str(dataset[0]["B"])

input_str = f"Question: {prompt_0_str}. Is the correct answer A: {opt_A_0} or B: {opt_B_0}? Answer with just the letter A or B."

inputs = tokenizer_t5(input_str, return_tensors="pt")
outputs = model_t5.generate(**inputs, max_new_tokens=5)
print(tokenizer_t5.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


B
